<a href="https://colab.research.google.com/github/DivineUI/lab-4-llm-decision-support/blob/main/lab_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 4: LLMs and Prompt Engineering for Decision Support

**Student Name:** Divine Uwase Ingabire
**Student ID:** 69062028

---

### Choosen an API provider

Provider :**Groq**


### Part 0: Repository and API-key setup


In [1]:
!pip install openai -q

In [2]:
# API-key setup
import os
from google.colab import userdata
API_KEY = userdata.get("GROQ_API_KEY")

# OpenAI-compatible client (works for Groq and OpenAI; Gemini users see their docs):
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",)
MODEL = "llama-3.3-70b-versatile"

print("Client ready.")

Client ready.


---
# Section 1 — Talking to an LLM Programmatically


### Part 1.1 — Your first API call

In [3]:
# Write a helper function you will reuse for the WHOLE lab:
def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
            temperature=0.7, max_tokens=500):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response, response.choices[0].message.content # Added return response to be able to compute token usage later

# Call it once with a simple question and print the answer.
response, answer = ask_llm("Explain computer vision in simple words")
print(answer)
print()
# Print response.usage as well
print("Token usage:", response.usage)

**Computer Vision: Seeing Like Humans**

Computer vision is a way to make computers "see" and understand the world like humans do. It's a field of study that helps computers interpret and make sense of visual information from images and videos.

**How it works:**

1. **Capture**: A computer captures an image or video from a camera, sensor, or other device.
2. **Process**: The computer breaks down the image or video into smaller parts, like shapes, colors, and textures.
3. **Analyze**: The computer uses algorithms (like recipes for computers) to analyze these parts and identify objects, patterns, and relationships.
4. **Understand**: The computer makes decisions or takes actions based on what it has analyzed, like recognizing a face, detecting an obstacle, or tracking movement.

**Examples:**

* Self-driving cars use computer vision to detect lanes, pedestrians, and traffic signals.
* Facebook uses computer vision to recognize faces and suggest tags in photos.
* Security cameras use com

**Student Reasoning — Anatomy of a call**
*1. What is the difference between the `system` and `user` roles? Give an example of
something that belongs in each.*
*2. What is a token, roughly? Why do API providers bill per token rather than per request?*

> **Answer:**


1.   **The system role** is where the developer of the application sets instructions on how the model should behave, the tone to use, security guidelines, and expected output format while **the user role** is the actual person using the app/website, giving it a specific task to do.
For example, the system message might say "You are a tutor for university students in computer science courses, help each student understand the concept in the most simple and concise way." while the user message would be the actual computer science text like "Explain system role and user role with examples."

2.   A token is the atomic unit a model reads, it is roughly a chunk of text.
Bill per token ties the price directly to the actual computational work. if the user were to be charged per request, it would be unfair for one-word questions like hi that don't cost much to answer to be charged the same as a task that requires to generate a 50 page document.



### Part 1.2 — Temperature: the randomness dial

In [4]:
question = "Suggest a name for a savings product for market traders in Accra."

print("     Temperature = 0.0")
for i in range(5):
    _, answer = ask_llm(question, temperature=0.0)
    print(f"\nResponse {i+1}")
    print(answer)

print("\n\n      Temperature = 1.2")
for i in range(5):
    _, answer = ask_llm(question, temperature=1.2)
    print(f"\nResponse {i+1}")
    print(answer)

     Temperature = 0.0

Response 1
Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **Trader's Treasure**: This name emphasizes the idea of saving and accumulating wealth.
3. **Accra Amanfu**: "Amanfu" is a Ghanaian word for "savings" or "treasury", so this name incorporates local language and culture.
4. **Market Mobi**: This name is short and catchy, and "Mobi" implies mobility and flexibility, which could appeal to market traders.
5. **Sika Kurom**: "Sika" means "money" in Ghanaian, and "Kurom" means "box" or "container", so this name suggests a safe and secure place to store savings.
6. **Traders' Fund**: This name is straightforward and emphasizes the idea of a collective fund for market traders.
7. **Adanfo Save**: "Adanfo" means "friends" or "partners" in Ghanaian, so this name suggests a sense of community and cooperation.

Choose the o

**Student Reasoning — Temperature**
*What did you observe at each temperature? For the loan decision-support system you are about
to build, which temperature regime is appropriate, and why?*

> **Answer:**
1.   If the few-shot example came from the six letters I am processing, it would cause bias; the model might base on that specific letter's numbers or phrasing and unconsciously pattern-match the other letters toward it.
Also, if the few-shot example came from one of those same six letters, it will be like showing the model the answer to one of the questions on its own test, so success on that letter wouldn't determine whether the technique works.

2. “use null, do not guess” matters bacause language models are designed to give complete and helpful answers. So, if some information is missing, the model may try to fill in the blank instead of saying “I don’t know.” which would be risky since the user wouldn't know that it is a made-up information.

3. The goal of Extraction is to get the exact information from the text.For example, the letter either says the repayment period is 6 months or it does not. The model should report what the letter actually says, not create a new answer.

Using `temperature=0` helps the model give the identical answer each time. Adding randomness can make the extraction less accurate or less consistent.

Creative tasks are different. For example, when coming up with product names, there is no single correct answer. Many different names can be good. In this case, variety is useful.

Using a higher temperature, such as `temperature=1.2`, encourages the model to generate different ideas each time. Using `temperature=0` would produce the same answer repeatedly, which is not helpful when the goal is brainstorming.


---
# Section 2 — The Dataset: Loan Application Letters


In [5]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.



# Section 3 — Prompt Engineering for the Decision Support System


### Part 3.1 — Component 1: Summarization


In [6]:
# SUMMARY_PROMPT_V1
SUMMARY_PROMPT_V1 = "Summarize this: {letter}"

for letter_id in ["L002", "L006"]:
    prompt = SUMMARY_PROMPT_V1.format(letter=LETTERS[letter_id])
    _, answer = ask_llm(prompt, temperature=0)
    print(f"   V1 — {letter_id}")
    print(answer)
    print()
    print()

# SUMMARY_PROMPT_V2
SUMMARY_SYSTEM_V2 = (
    "You are an assistant to a microfinance loan officer. Summarize loan application "
    "letters factually and neutrally. Do not invent details that are not stated in the letter."
     "Do not add opinions, sympathy, or recommendations. Write exactly 3-4 sentences.")

SUMMARY_PROMPT_V2 = "Summarize this loan application:\n\n{letter}"

for letter_id in ["L002", "L006"]:
    prompt = SUMMARY_PROMPT_V2.format(letter=LETTERS[letter_id])
    _, answer = ask_llm(prompt, system_prompt=SUMMARY_SYSTEM_V2, temperature=0)
    print(f"   V2 — {letter_id}")
    print(answer)
    print()
    print()

# TODO: Compare V1 vs V2 outputs side by side. Keep both prompt versions in this notebook.
for letter_id in ["L002", "L006"]:
    print(f"               Letter {letter_id}")
    prompt_v1 = SUMMARY_PROMPT_V1.format(letter=LETTERS[letter_id])
    _, v1 = ask_llm(prompt_v1, temperature=0)
    prompt_v2 = SUMMARY_PROMPT_V2.format(letter=LETTERS[letter_id])
    _, v2 = ask_llm(prompt_v2, system_prompt=SUMMARY_SYSTEM_V2, temperature=0)
    print("     V1")
    print(v1)
    print("\n        V2")
    print(v2)
    print()
    print()

   V1 — L002
Kwame Boateng, a commercial driver in Kumasi, is urgently seeking GHS 25,000 to repair his vehicle's engine and pay off personal debts. He's experiencing a slow business period but expects it to improve after the festive season. He has no collateral to offer but promises to repay the loan as soon as possible.


   V1 — L006
Kofi, a 22-year-old, is seeking a loan of GHS 50,000 to start three businesses: a car washing service, a provision shop, and a phone import business from Dubai. He has no prior experience, but claims to be "business-minded" based on his friends' opinions. He promises to repay the loan within a year, once his businesses are successful, and offers no collateral, relying on his personal trustworthiness.


   V2 — L002
Kwame Boateng, a commercial driver in Kumasi, has applied for a loan of GHS 25,000. He intends to use the funds to repair his trotro engine and settle personal debts. Mr. Boateng mentions that his business has been slow, but expects it to imp

**Student Reasoning — Summarization prompts**
*1. What concrete problems did V1's output have that V2 fixed? Quote examples.*
*2. Why is "no invented details" an essential instruction in this application? What is this
failure mode called in the LLM literature?*

> **Answer:**
1. V1 sometimes made vague statements sound more confident than they really were. For L002, V1 said Kofi “promises to repay the loan as soon as possible,” but the letter says, “I can pay back whenever the money comes.” Showing that the original statement is less certain and shows more risk. For L006, V1 said Kofi “has no prior experience,” while V2 was able to solve this issue and provide more important information, for example, more clearly stated that “Kofi has not yet started any of these businesses.” This provides important information to the loan officer because it directly shows that the businesses have not yet started.

2. “No invented details” is important because loan officers use these summaries to make real financial decisions. If a summary adds confidence or leaves out important risks, the officer could approve a bad loan based on an inaccurate information. In LLM research, this is called hallucination. When the problem occurs while summarizing a real source document, it is often called factual consistency failure means the summary sounds clear and believable drifts from what the source text actually supports.

### Part 3.2 — Component 2: Structured extraction (JSON)


In [7]:
# TODO: Write EXTRACT_PROMPT
import pandas as pd
EXTRACT_SYSTEM = ("You are a data extraction assistant for a microfinance loan officer. "
    "Extract ONLY the fields requested, as strict JSON. Do not include any text "
    "before or after the JSON object, and do not use markdown code fences.")

EXTRACT_PROMPT = """Extract the following fields from the loan application letter below and return ONLY a JSON object with exactly these keys:
applicant_name (string)
amount_ghs (number)
purpose (string)
monthly_profit_ghs (number or null)
has_collateral_or_guarantor (boolean)
repayment_months (number or null)
If a field is not stated in the letter, use null. Do not guess or infer a value that is not explicitly stated.
Example:
Letter:
"My name is Divine Uwase. I run a medium bakery in Accra and I am requesting GHS 10,000 to buy a new oven. I make about GHS 1000 profit a month. I have no guarantor yet but I am hoping to find one. I can repay over 10 months."
JSON:
{{"applicant_name": "Divine Uwase", "amount_ghs": 10,000, "purpose": "buy a new oven", "monthly_profit_ghs": 1000, "has_collateral_or_guarantor": false, "repayment_months": 10}}
Now extract from this letter:
{letter}
JSON:"""

# extract_fields(letter_text) function
import json
import re

def extract_fields(letter_text):
    prompt = EXTRACT_PROMPT.format(letter=letter_text)
    _, raw = ask_llm(prompt, system_prompt=EXTRACT_SYSTEM, temperature=0, max_tokens=300)

    cleaned = re.sub(r"^```(?:json)?\s*|\s*```$", "", raw.strip(), flags=re.MULTILINE).strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError as e:
        print(f"Warning: failed to parse JSON. Error: {e}\nRaw output: {raw}")
        return None
# Running on all six letters and building a DataFrame
results = {}
for letter_id, letter_text in LETTERS.items():
    fields = extract_fields(letter_text)
    results[letter_id] = fields

df = pd.DataFrame.from_dict(results, orient="index")
df.index.name = "letter_id"
df

,applicant_name,amount_ghs,purpose,monthly_profit_ghs,has_collateral_or_guarantor,repayment_months
letter_id,,,,,,
L001,Akosua Mensah,8000,buy a deep freezer and expand into frozen foods,900.0,True,20.0
L002,Kwame Boateng,25000,repair my trotro engine and settle some person...,NaN,False,NaN
L003,Efua Darko,15000,purchase two industrial sewing machines and fa...,2800.0,True,15.0
L004,Yaw Owusu,12000,for feed and 500 new layers,1500.0,True,18.0
L005,Adenta Women's Weaving Cooperative,30000,buy a bulk order of yarn directly from the fac...,NaN,True,16.0
L006,Kofi,50000,"start a car washing business, a provision shop...",NaN,False,12.0


In [8]:
print(GOLD)

{'L001': {'applicant_name': 'Akosua Mensah', 'amount_ghs': 8000, 'purpose': 'buy deep freezer / expand into frozen foods', 'monthly_profit_ghs': 900, 'has_collateral_or_guarantor': True, 'repayment_months': 20}, 'L003': {'applicant_name': 'Efua Darko', 'amount_ghs': 15000, 'purpose': 'industrial sewing machines and fabric stock', 'monthly_profit_ghs': 2800, 'has_collateral_or_guarantor': True, 'repayment_months': 15}, 'L006': {'applicant_name': 'Kofi', 'amount_ghs': 50000, 'purpose': 'car wash, provision shop, phone imports', 'monthly_profit_ghs': None, 'has_collateral_or_guarantor': False, 'repayment_months': 12}}


**Student Reasoning — Structured extraction**
*1. Why must the few-shot example NOT come from the six letters you are processing?*
*2. Why "use null, do not guess" — what did the model do without that instruction?*
*3. Why is temperature=0 the right choice for extraction but arguably not for creative tasks?*

> **Answer:**

1. The few-shot example should not come from the six letters being tested because using one of them could leak test information into the model's reference example and affect how it reads the other letters. It would also make the result inaccurate evaluation because the model would already have seen the answer.

2. Without the instruction to **“use null, do not guess,”** a model may fill in a missing value with a plausible-looking number instead of admitting that the information is not provided. But with this instruction, L006 correctly returned `monthly_profit_ghs: null` because Kofi's letter does not state a profit figure. This exactly matched the gold label and shows that the instruction has a practical effect.

3. Temperature = 0 is appropriate for extraction because each field has one correct answer based on the letter. For example, the letter either states monthly profit or it does not. Randomness can therefore lead to inconsistent or incorrect results without providing any benefit. Creative tasks is used when variation is useful.




### Part 3.3 — Component 3: The decision-support brief

In [9]:
# TODO: Write BRIEF_PROMPT
BRIEF_SYSTEM = ("You are an assistant to a microfinance loan officer in Ghana. Your job is to help "
    "the officer make an informed decision — you do NOT make the decision yourself. "
    "Never output a recommendation to approve or reject the loan. Base every point "
    "strictly on the letter and extracted data provided; do not invent facts.")

BRIEF_PROMPT = """Given the loan application letter and the extracted data below, produce a decision-support brief with exactly these four sections:
1. Strengths (bullet points, grounded only in the letter/data)
2. Risks / Red flags (bullet points)
3. Missing information the officer should request before deciding
4. Suggested next step — choose ONE of: "invite for interview", "request documents", "flag for senior review". Do NOT say "approve" or "reject" — the final decision is made by a human loan officer, not by you.

Letter:
{letter}

Extracted data (JSON):
{extracted_json}

Brief:"""

# Generate briefs for ALL SIX letters. Print the briefs for L001, L002, and L006
import json as json_lib

briefs = {}
for letter_id, letter_text in LETTERS.items():
    extracted_json = json_lib.dumps(results[letter_id])
    prompt = BRIEF_PROMPT.format(letter=letter_text, extracted_json=extracted_json)
    _, brief = ask_llm(prompt, system_prompt=BRIEF_SYSTEM, temperature=0, max_tokens=500)
    briefs[letter_id] = brief

for letter_id in ["L001", "L002", "L006"]:
    print(f"         Letter {letter_id}")
    print(briefs[letter_id])
    print()

         Letter L001
## Step 1: Strengths
* The applicant, Akosua Mensah, has 12 years of experience selling provisions at Makola Market, indicating a stable business history.
* She has a proven track record of saving with the susu scheme, having saved GHS 2,500 over two years without missing a contribution.
* The applicant has a guarantor, her sister, who is a teacher, providing an added layer of security for the loan.
* Akosua Mensah has a clear plan for using the loan, which is to buy a deep freezer and expand into frozen foods, potentially increasing her business income.

## Step 2: Risks / Red flags
* The loan amount of GHS 8,000 is significant compared to her monthly profit of GHS 900, which might pose a risk if her business does not generate enough income to cover the loan repayments.
* The repayment plan of GHS 450 monthly over 20 months is relatively high compared to her current monthly profit, which could strain her financial resources.
* There is no detailed information prov

In [10]:
print(briefs["L003"])

## Step 1: Strengths
* The business, Darko Fashions, is registered, indicating a level of legitimacy and compliance with regulatory requirements.
* The applicant, Efua Darko, has a fixed deposit of GHS 5,000 that can be pledged as collateral, reducing the risk for the lender.
* The business has a history of generating significant revenue, with GHS 22,000 in December revenue last year, and an average monthly profit of GHS 2,800.
* The applicant has provided sales records for the past 18 months, which can help in assessing the business's financial health and stability.

## Step 2: Risks / Red flags
* The loan amount of GHS 15,000 is substantial, and the repayment plan of GHS 1,100 monthly for 15 months needs to be carefully evaluated to ensure it is manageable for the business.
* The proposal is based on anticipated demand ahead of the Christmas season, which may not materialize as expected, posing a risk to the business's ability to repay the loan.
* There is no detailed information on 

**Student Reasoning — Decision support**
*1. Compare the briefs for L003 (strong application) and L006 (weak application). Did the
system identify the right strengths and red flags in each?*
*2. Why did we forbid the model from outputting "approve"/"reject"? Give one practical and
one ethical reason.*

> **Answer:**
1. Yes, the system correctly distinguished a strong application from a weak one. For L003, the strengths were specific and verifiable; the business was registered, GHS 5,000 in fixed-deposit collateral, GHS 22,000 in December revenue, and 18 months of sales records. The risks were also reasonable; seasonal income and no clear credit history. For L006 (Kofi), the system correctly recognized that there is no operating business yet. Its listed strengths were weak such as being full of energy and considered “business minded” by friends. Also, the risks were more serious; no experience, no collateral, and a repayment plan based on the assumption that the businesses will boom within a year. Overall, the system correctly distinguished evidence and proven activity (L003) from aspirations and unproven plans (L006).

2. The reason why we have forbidden the model from outputting "approve"/"reject" is because the model can be confidently wrong. It does not have all the information needed for a real lending decision, such as credit history, verified documents, regulatory requirements, or the lender’s risk limits. An “approve/reject” answer could therefore sound authoritative while being based on incomplete information. Also, decisions can since the output affects someone’s finances and livelihood, a human should remain responsible for the final decision. If loan officers simply accept an LLM’s recommendation, even unintentionally, accountability could be lost.

### Part 3.4 — Commit your prompt templates


> **Commit hash:** 080cc3f7dc367e7c0cd148886724539080951243

---
# Section 4 — Evaluation: Quality, Reliability, Appropriateness


### Part 4.1 — Extraction accuracy against gold labels

In [11]:
# TODO: For the three letters in GOLD, compare your extracted DataFrame to the gold values
gold_letters = ["L001", "L003", "L006"]
fields = ["applicant_name", "amount_ghs", "purpose", "monthly_profit_ghs",
          "has_collateral_or_guarantor", "repayment_months"]

# Display a small table: rows = fields, columns = L001 / L003 / L006 / accuracy.
comparison = {}
for field in fields:
    row = {}
    correct_count = 0
    for letter_id in gold_letters:
        gold_val = GOLD[letter_id][field]
        extracted_val = df.loc[letter_id, field]

        # normalize for comparison
        if field == "applicant_name":
            match = str(gold_val).strip().lower() == str(extracted_val).strip().lower()
        elif isinstance(gold_val, float) and pd.isna(gold_val):
            match = pd.isna(extracted_val)
        elif gold_val is None:
            match = pd.isna(extracted_val) or extracted_val is None
        else:
            match = gold_val == extracted_val

        row[letter_id] = "✓" if match else f"✗ (got {extracted_val})"
        if match:
            correct_count += 1

    row["accuracy"] = f"{correct_count}/3"
    comparison[field] = row

accuracy_df = pd.DataFrame.from_dict(comparison, orient="index")
accuracy_df


,L001,L003,L006,accuracy
applicant_name,✓,✓,✓,3/3
amount_ghs,✓,✓,✓,3/3
purpose,✗ (got buy a deep freezer and expand into froz...,✗ (got purchase two industrial sewing machines...,"✗ (got start a car washing business, a provisi...",0/3
monthly_profit_ghs,✓,✓,✓,3/3
has_collateral_or_guarantor,✓,✓,✓,3/3
repayment_months,✓,✓,✓,3/3


### Part 4.2 — Reliability: is the system consistent?

In [12]:
def extract_fields(letter_text, temperature=0):
    prompt = EXTRACT_PROMPT.format(letter=letter_text)
    _, raw = ask_llm(prompt, system_prompt=EXTRACT_SYSTEM, temperature=temperature, max_tokens=300)

    cleaned = re.sub(r"^```(?:json)?\s*|\s*```$", "", raw.strip(), flags=re.MULTILINE).strip()

    try:
        return json.loads(cleaned)
    except json.JSONDecodeError as e:
        print(f"Warning: failed to parse JSON. Error: {e}\nRaw output: {raw}")
        return None

temp0_runs = [extract_fields(LETTERS["L004"], temperature=0) for _ in range(5)]
temp1_runs = [extract_fields(LETTERS["L004"], temperature=1.0) for _ in range(5)]

def summarize_runs(runs, label):
    valid_json = sum(1 for r in runs if r is not None)
    signatures = [json.dumps(r, sort_keys=True) for r in runs if r is not None]
    unique_signatures = set(signatures)
    print(f"          {label}")
    print(f"Valid JSON:{valid_json}/5")
    print(f"Unique results: {len(unique_signatures)} (1 = fully consistent)")
    for i, r in enumerate(runs):
        print(f"  Run {i+1}: {r}")
    print()

summarize_runs(temp0_runs, "Temperature=0.0")
summarize_runs(temp1_runs, "Temperature=1.0")

          Temperature=0.0
Valid JSON:5/5
Unique results: 1 (1 = fully consistent)
  Run 1: {'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'for feed and 500 new layers', 'monthly_profit_ghs': 1500, 'has_collateral_or_guarantor': True, 'repayment_months': 18}
  Run 2: {'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'for feed and 500 new layers', 'monthly_profit_ghs': 1500, 'has_collateral_or_guarantor': True, 'repayment_months': 18}
  Run 3: {'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'for feed and 500 new layers', 'monthly_profit_ghs': 1500, 'has_collateral_or_guarantor': True, 'repayment_months': 18}
  Run 4: {'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'for feed and 500 new layers', 'monthly_profit_ghs': 1500, 'has_collateral_or_guarantor': True, 'repayment_months': 18}
  Run 5: {'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'for feed and 500 new layers', 'monthly_profit_ghs': 1500, 'has_collatera

### Part 4.3 — Hallucination probing

In [13]:
# TODO: Design TWO adversarial tests and run them:
#   Test 1 — Ask your summarizer a question about a detail that is NOT in a letter
adversarial_prompt_1 = "What is the applicant's credit score, according to this letter?\n\n" + LETTERS["L002"]

_, test1_answer = ask_llm(adversarial_prompt_1, system_prompt=SUMMARY_SYSTEM_V2, temperature=0)
print("Test 1: asking for a detail NOT in the letter")
print(test1_answer)

#   Test 2 — Feed your extractor an EMPTY or IRRELEVANT text
weather_report = ("Weather Report for Accra, Ghana — Tuesday\n"
    "Today expect partly cloudy skies with a high of 31°C and a low of 24°C. "
    "Humidity will be around 78%, with a light breeze from the southwest at "
    "12 km/h. Chance of rain in the afternoon is 40%. UV index: high.")

test2_result = extract_fields(weather_report, temperature=0)
print(" \nTest 2: extracting from an irrelevant document")
print(test2_result)


Test 1: asking for a detail NOT in the letter
The letter does not mention the applicant's credit score. Kwame Boateng is requesting a loan of GHS 25,000 to repair his trotro engine and settle personal debts. He mentions that business has been slow, but expects it to improve after the festive season. The applicant does not provide any information about his credit history or financial records.
 
Test 2: extracting from an irrelevant document
{'applicant_name': None, 'amount_ghs': None, 'purpose': None, 'monthly_profit_ghs': None, 'has_collateral_or_guarantor': None, 'repayment_months': None}


### TODO: Record the outputs verbatim below and label each PASS or FAIL.
**Test 1 output :** The letter does not mention the applicant’s credit score. Kwame Boateng is requesting a loan of GHS 25,000 to repair his trotro engine and pay personal debts. He says that business has been slow, but he expects it to improve after the festive season. The applicant does not provide any information about his credit history or financial records.”

**Label:** I would lebel this as **PASS** because the model correctly said that the credit score was not mentioned instead of making up a number. It stayed focused on the information that is provided in the letter.

**Test 1 output :** 'applicant_name': None, 'amount_ghs': None, 'purpose': None, 'monthly_profit_ghs': None, 'has_collateral_or_guarantor': None, 'repayment_months': None

**Label:  I would lebel this as **PASS** because the weather report was not related to loans and the model correctly recognized that there was no applicant information to extract.

**Student Reasoning — Evaluation results**
*1. Report your extraction accuracy. Which field was hardest for the model and why?*
*2. What did the reliability experiment show about temperature and production systems?*
*3. Did your system hallucinate under probing? If yes, how could the prompt (or the system
design around it) reduce the risk?*

> **Answer:**

1.

* Extraction Accuracy: The model achieved 100% accuracy (3/3) on most fields across the gold-standard letters: `applicant_name`, `amount_ghs`, `monthly_profit_ghs`, `has_collateral_or_guarantor`, and `repayment_months`.

* The purpose field scored 0/3 in the automated string-matching test because the model used different wording from the gold label. For example, "buy a deep freezer and expand into frozen foods" and "buy deep freezer / expand into frozen foods" have the same meaning, but are not the exact text match. So, the purpose field was the most challenging because applicants describe their loan purpose in different, natural language. Unlike structured fields such as `amount_ghs`, so for `purpose` exact string matching may mark a correct answer as wrong even when the meaning is accurate.

2.  When the same letter (`L004`) was tested 5 times at **Temperature = 0.0** and **Temperature = 1.0**, the model produced 5/5 valid JSON outputs with one unique result in both cases. This means the results were fully consistent, even at a higher temperature unlike our previous observation on part 1.2.
For production decision-support systems, Temperature = 0 is not strictly necessary for consistency on clean inputs but is still recommended because it reduces randomness and makes data extraction more predictable, consistent, and reproducible.

3. No, the system did not hallucinate under either test.



### Part 4.4 — Appropriateness: should this system exist?


**Student Reasoning — Appropriateness**
*1. Letters L002 and L006 would likely be declined. If the bank fully automated decisions
with your system, who could be unfairly harmed, and how? Consider applicants who write
poorly in English but run solid businesses.*
*2. Loan letters contain personal data. What are the implications of sending them to a
third-party API in another country? What would you check before deploying this at a real
Ghanaian microfinance institution?*
*3. Name TWO concrete safeguards you would build around this system in production (think:
human review points, logging, appeal processes, monitoring).*


> **Answer:**

1. If a bank fully automated loan decisions using this system, applicants with skills and potential but struggle to write well in English would be unfairly harmed. Letters like L002, Kwame Boateng, a commercial driver, and L006, Kofi, starting multiple ventures are informal and unstructured. So, their application might get rejected because automated LLM extraction and summarization pipeline relies heavily on clear, professional phrasing and structured like fixed assets or formal accounting records.

2. Sending loan applications with sensitive personal details to a third-party API in another country can create privacy, security, and regulatory risks. So, I would ensure that financial institutions ensure compliance with laws and other cross-border data transfer requirements, and before deployment, I would verify customer consent requirements, encryption, data-use policies, and security standards.


3.
 * Every generated output must be used as a suggestion or advisory output not the final decision; and a human loan officer must verify the output and make the final final approval or rejection decision.

 * Record every prompt and output, so the institution can check whether the outputs are the expected and also that if a loan decision is questioned, they can be able to trace it.

 4. Using an API is better for this task because training a model would require a large labeled dataset, and there is only 6 letters and 3 gold labels available for task. It is also faster and easier to use an already-trained model for tasks like summarizing, extraction, and reasoning.

Training a custom model would be better if there were a large labeled dataset, a narrow and well-defined task, or when I needed full control over a model.





---
### Submission checklist

- [ ] All cells run top-to-bottom with no errors (`Kernel -> Restart & Run All`).
- [ ] **No API key anywhere in the notebook or the commit history.**
- [ ] Every **Student Reasoning** box is filled in with full sentences.
- [ ] `prompts.py` / `prompts.md` committed with your final prompt templates.
- [ ] Evaluation tables and adversarial test outputs visible in the saved notebook.
- [ ] Notebook pushed to `lab-4-llm-decision-support` with incremental commits.
- [ ] Repository link submitted to the course portal.
- [ ] AI Declaration form in Repository.